# 🧠 Filtrado Avanzado, Consultas y Exportación de Datos

En el módulo anterior aprendimos a hacer filtros usando "máscaras booleanas" (ej. `df[df['Columna'] == 'Valor']`). Sin embargo, a medida que las preguntas de negocio se vuelven más complejas, ese código se vuelve difícil de leer.

En este notebook daremos un salto de nivel trabajando con un dataset de **Atenciones de Partos en Establecimientos de Salud (MINSA)**. Aprenderemos a:
1. Navegar por posiciones exactas usando `iloc`.
2. Hacer consultas estilo SQL usando `query()`.
3. Cruzar filtros avanzados con `loc` y limitar resultados con `head()`.
4. Exportar nuestros hallazgos a un nuevo archivo con `to_csv()`.

In [5]:
import pandas as pd

# Cargamos el dataset. Usamos delimiter=';' ya que es el estándar en muchos sistemas.
df_cons = pd.read_csv('DM02_ConsultaE.csv', delimiter=';')

print("Columnas disponibles:", df_cons.columns.tolist())
df_cons.head()

Columnas disponibles: ['AÑO', 'MES', 'UBIGEO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SECTOR', 'CATEGORIA', 'CO_IPRESS', 'RAZON_SOC', 'ID_PARTO', 'TIPO_PARTO', 'PARTOS_TOTAL', 'PARTOS_VIVOS', 'PARTOS_MUERTOS']


,AÑO,MES,UBIGEO,DEPARTAMENTO,PROVINCIA,DISTRITO,SECTOR,CATEGORIA,CO_IPRESS,RAZON_SOC,ID_PARTO,TIPO_PARTO,PARTOS_TOTAL,PARTOS_VIVOS,PARTOS_MUERTOS
0,2026,4,160101,LORETO,MAYNAS,IQUITOS,MINSA,II-2,1,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",2,CESAREA,58,52,6
1,2026,4,160101,LORETO,MAYNAS,IQUITOS,MINSA,II-2,1,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",2,CESAREA,4,4,0
2,2026,4,160101,LORETO,MAYNAS,IQUITOS,MINSA,II-2,1,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",1,VAGINAL,161,161,0
3,2026,3,160101,LORETO,MAYNAS,IQUITOS,MINSA,II-2,1,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",2,CESAREA,70,66,4
4,2026,3,160101,LORETO,MAYNAS,IQUITOS,MINSA,II-2,1,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",2,CESAREA,3,3,0


## Selección por Posición: El método `iloc`

A diferencia de `loc` (que busca por el *nombre* de la etiqueta o índice), **`iloc`** busca exclusivamente por la **posición numérica entera** (integer location). Es como acceder a una matriz matemática clásica.

**Sintaxis:** `df.iloc[filas, columnas]`

In [6]:
# 1. Extraer las primeras 3 filas y solo las primeras 4 columnas
print("--- Filas 0 a 2 | Columnas 0 a 3 ---")
display(df_cons.iloc[0:3, 0:4])

# 2. Extraer filas salteadas (ej. fila 0, 5, 10) y columnas específicas por posición (Departamento, Provincia, Partos Vivos)
print("\n--- Selección específica por índices numéricos ---")
display(df_cons.iloc[[0, 5, 10], [3, 4, 13]])

--- Filas 0 a 2 | Columnas 0 a 3 ---


,AÑO,MES,UBIGEO,DEPARTAMENTO
0,2026,4,160101,LORETO
1,2026,4,160101,LORETO
2,2026,4,160101,LORETO



--- Selección específica por índices numéricos ---


,DEPARTAMENTO,PROVINCIA,PARTOS_VIVOS
0,LORETO,MAYNAS,52
5,LORETO,MAYNAS,162
10,LORETO,MAYNAS,6


## Consultas Elegantes con `query()`

Cuando tienes múltiples condiciones, el filtrado tradicional con corchetes se vuelve ruidoso por la repetición de `df[...]`. El método `.query()` te permite escribir expresiones lógicas como si fueran sentencias SQL, haciendo el código increíblemente limpio.

In [16]:
# Preguntas de negocio:
# A. ¿Cuáles son los hospitales en LORETO donde hubo más de 50 partos por CESÁREA en el año 2026?
# B. ¿Cuáles fueron los hospitales del departamento de Loreto que acumularon más de 300 partos por cesárea durante el año 2026?

# Convertimos las columnas numéricas que fueron importadas como texto
df_cons["PARTOS_TOTAL"] = pd.to_numeric(df_cons["PARTOS_TOTAL"], errors="coerce")
df_cons["PARTOS_VIVOS"] = pd.to_numeric(df_cons["PARTOS_VIVOS"], errors="coerce")
df_cons["PARTOS_MUERTOS"] = pd.to_numeric(df_cons["PARTOS_MUERTOS"], errors="coerce")

# ============================================================================
# A. Hospitales que registraron más de 50 partos por cesárea en algún mes de 2026
# ============================================================================

# Filtramos los registros que cumplen con las condiciones de la consulta
hospitales_cesarea_a = (
    df_cons.query(
        "DEPARTAMENTO == 'LORETO' and TIPO_PARTO == 'CESAREA' and PARTOS_TOTAL > 50 and AÑO == 2026"
    )[["RAZON_SOC"]]
    .drop_duplicates()
)

print("\n--- A. Hospitales que registraron más de 50 partos por cesárea en algún mes del año 2026 ---")
display(hospitales_cesarea_a)


# ============================================================================
# B. Hospitales que acumularon más de 50 partos por cesárea durante todo el año 2026
# ============================================================================

# Filtramos únicamente los registros de cesáreas en Loreto durante el año 2026
cesareas_loreto = df_cons.query(
    "DEPARTAMENTO == 'LORETO' and TIPO_PARTO == 'CESAREA' and AÑO == 2026"
)

# Agrupamos por hospital y sumamos el total de partos por cesárea registrados durante el año
hospitales_cesarea_b = (
    cesareas_loreto
    .groupby("RAZON_SOC", as_index=False)["PARTOS_TOTAL"]
    .sum()
)

# Conservamos solo los hospitales que acumularon más de 300 cesáreas
hospitales_cesarea_b = hospitales_cesarea_b.query("PARTOS_TOTAL > 300")

# Ordenamos los resultados de mayor a menor
hospitales_cesarea_b = hospitales_cesarea_b.sort_values(
    by="PARTOS_TOTAL",
    ascending=False
)

print("\n--- B. Hospitales que acumularon más de 300 partos por cesárea durante el año 2026 ---")
display(hospitales_cesarea_b)


--- A. Hospitales que registraron más de 50 partos por cesárea en algún mes del año 2026 ---


,RAZON_SOC
0,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA"""
15,"HOSPITAL REGIONAL DE LORETO ""FELIPE SANTIAGO A..."
146,HOSPITAL II-2 SANTA GEMA DE YURIMAGUAS
24260,HOSPITAL III IQUITOS



--- B. Hospitales que acumularon más de 300 partos por cesárea durante el año 2026 ---


,RAZON_SOC,PARTOS_TOTAL
7,"HOSPITAL REGIONAL DE LORETO ""FELIPE SANTIAGO A...",406.0
4,HOSPITAL II-2 SANTA GEMA DE YURIMAGUAS,351.0
6,"HOSPITAL IQUITOS ""CESAR GARAYAR GARCIA""",329.0


## Precisión Quirúrgica: Filtros con `loc`

Mientras que `query()` es genial para leer el DataFrame completo, **`loc`** brilla cuando queremos filtrar filas pero **devolver solo columnas específicas** al mismo tiempo.

In [17]:
# Queremos ver solo el nombre del hospital (RAZON_SOC) y los partos muertos
# para los casos donde los partos muertos sean mayores a 0 en hospitales del MINSA.

mascara_fallecidos = (df_cons['PARTOS_MUERTOS'] > 0) & (df_cons['SECTOR'] == 'MINSA')

# Sintaxis de loc: df.loc[ CONDICION_DE_FILAS, LISTA_DE_COLUMNAS ]
reporte_fallecidos = df_cons.loc[mascara_fallecidos, ['RAZON_SOC', 'DEPARTAMENTO', 'PARTOS_TOTAL', 'PARTOS_MUERTOS']]

# Usamos head() para mostrar un resumen de los peores escenarios (ordenando los datos previamente)
reporte_fallecidos.sort_values(by='PARTOS_MUERTOS', ascending=False).head()

,RAZON_SOC,DEPARTAMENTO,PARTOS_TOTAL,PARTOS_MUERTOS
17078,INSTITUTO NACIONAL MATERNO PERINATAL,LIMA,218.0,12.0
17070,INSTITUTO NACIONAL MATERNO PERINATAL,LIMA,240.0,8.0
21146,HOSPITAL NACIONAL SERGIO E. BERNALES,LIMA,90.0,7.0
17065,INSTITUTO NACIONAL MATERNO PERINATAL,LIMA,502.0,7.0
21143,HOSPITAL NACIONAL SERGIO E. BERNALES,LIMA,90.0,7.0


## Guardando el Trabajo: `to_csv()`

Una vez que hemos procesado, filtrado y limpiado la información, el último paso en el flujo de trabajo de análisis de datos es exportar los resultados para poder compartirlos (por ejemplo, con un equipo de Business Intelligence que usa Power BI o Excel).

In [18]:
# Vamos a guardar el reporte de fallecidos que generamos en el paso anterior.
# El parámetro 'index=False' es vital para no guardar la columna de números (0, 1, 2...) que Pandas genera automáticamente.

nombre_archivo = 'MD02_Reporte_Partos_Muertos_MINSA.csv'

reporte_fallecidos.to_csv(nombre_archivo, index=False, sep=';', encoding='utf-8')

print(f"¡El archivo '{nombre_archivo}' se ha guardado correctamente en tu directorio actual!")

¡El archivo 'MD02_Reporte_Partos_Muertos_MINSA.csv' se ha guardado correctamente en tu directorio actual!
